In [0]:
%sql
USE CATALOG retail_sales_catalog;
USE SCHEMA silver;

Clean Customers

In [0]:
%sql
USE CATALOG retail_sales_catalog;
USE SCHEMA bronze;
CREATE OR REPLACE TABLE silver.customers_clean AS
SELECT 
    CustomerID,
    CustomerName,
    LOWER(TRIM(Email)) AS Email,
    TRIM(City) AS City,
    TRIM(Address) AS Address,
    LastUpdated
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY CustomerID
               ORDER BY LastUpdated DESC
           ) AS rn
    FROM bronze.customers_raw
) t
WHERE rn = 1;

Clean Products

In [0]:
%sql
CREATE TABLE silver.products_clean AS
SELECT DISTINCT
    ProductID,
    TRIM(ProductName) AS ProductName,
    TRIM(Category) AS Category,
    UnitPrice
FROM bronze.products_raw;

Clean Store

In [0]:
%sql
CREATE TABLE silver.stores_clean AS
SELECT DISTINCT
    StoreID,
    TRIM(StoreName) AS StoreName,
    TRIM(Region) AS Region
FROM bronze.stores_raw;

Clean Sales

In [0]:
%sql
CREATE OR REPLACE TABLE silver.sales_clean AS
SELECT DISTINCT
    TransactionID,
    CustomerID,
    ProductID,
    StoreID,
    Quantity,
    TxnDate
FROM bronze.sales_raw
WHERE Quantity > 0;

In [0]:
%sql
CREATE OR REPLACE TABLE customers_clean AS
SELECT DISTINCT
    CustomerID,
    CustomerName,
    LOWER(TRIM(Email)) AS Email,
    TRIM(City) AS City,
    TRIM(Address) AS Address
FROM bronze.customers_copy_raw;

Incremental 

In [0]:
%sql
USE CATALOG retail_sales_catalog;
USE SCHEMA silver;

CREATE OR REPLACE TABLE customers_clean AS
SELECT 
    CustomerID,
    CustomerName,
    LOWER(TRIM(Email)) AS Email,
    TRIM(City) AS City,
    TRIM(Address) AS Address,
    LastUpdated
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY CustomerID
               ORDER BY LastUpdated DESC
           ) rn
    FROM (
        SELECT 
            CustomerID,
            CustomerName,
            Email,
            City,
            Address,
            LastUpdated
        FROM retail_sales_catalog.bronze.customers_raw
        UNION ALL
        SELECT 
            CustomerID,
            CustomerName,
            Email,
            City,
            Address,
            TRY_CAST(TO_DATE(LastUpdated, 'dd-MM-yyyy') AS DATE) AS LastUpdated
        FROM retail_sales_catalog.bronze.customers_incremental_raw
    ) t
) x
WHERE rn = 1;